In [1]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import scb_multiome as scbm 
import scanpy as sc 
import numpy as np 
import matplotlib.pyplot as plt 
import json
import pandas as pd
import jax
from tqdm import tqdm
import flax 
import jax.numpy as jnp
from matplotlib import gridspec
import h5py
from scipy.stats import spearmanr 
import pickle5 

2025-05-30 15:05:27.282398: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.9.41). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [2]:
BASE_DIR = "/mnt/fxcai/nfs_share2"
data_dir = "../../data"
atac_path = f"{data_dir}/atac_data/ad_atac_132k.h5ad"
rna_path = f"{data_dir}/rna_data/tfs.h5ad"
preprocess_folder_acc = f"{data_dir}/atac_data/scb_processed"
preprocess_folder_rna = f"{data_dir}/rna_data"
chippath = f"{data_dir}/chip_data/atac_132k_encode_byTF.csv"
gt_path = f"{data_dir}/chip_data/blood_gt.csv"

result_dir = "."

In [3]:
ad_atac = sc.read_h5ad(atac_path)
ad_rna = sc.read_h5ad(rna_path)
sc.pp.scale(ad_rna)

chip_bulk = pd.read_csv(chippath, index_col=0)
df_gt = pd.read_csv(gt_path, index_col=0)


In [7]:
with open(f"{result_dir}/config.json") as f:
    config_d = json.load(f)
    
model = scbm.core.scb_TFs.Model(preprocess_folder_acc = preprocess_folder_acc,
                                preprocess_folder_rna=preprocess_folder_rna,
                                atac = ad_atac.X.T,
                                rna = ad_rna.X,
                                model_config=config_d['model_config']
                                )
model.create_model()
all_ds = model.read_accrna_ds(ds_key='all')

with open(f"{result_dir}/result.pkl", "rb") as handle:
    data = pickle5.load(handle)
model.load_data(data,
                keys = ['params_all', 'y_pred', 'metrics'])

In [ ]:
chip_path = "../data/chip_data/blood_gt.csv"

In [ ]:
jaspar_motifs = scbm.pp.read_JASPAR_pwms(f"{BASE_DIR}/genomes/JASPAR_human_TFs_meme/20230424043428_JASPAR2022_combined_matrices_2028_meme.txt")
jaspar_motifs[:5]
jaspar_motifs = jaspar_motifs[jaspar_motifs['tf'].isin([tf_ct.split("_")[0] for tf_ct in tfs_cts])]
jaspar_motifs['motif'].unique()